# MAIN

## Imports

In [37]:
#imports

import pandas as pd
import numpy as np 

#for importing the preprocessor done in the preprocessing folder
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessor, orig_onehot_enc_cols, orig_target_enc_cols, originaldf_preprocessor





from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer, cohen_kappa_score


## Dataset splitting

In [ ]:
# DON'T EXECUTE AGAIN, THE SPLIT IS ALREADY DONE AND CAN BE LOADED USING ITS SPECIFIC FILES
'''
df= pd.read_parquet("../data/cleaned/cleaned_df_final.parquet")
#print(df.columns.tolist())

#1. Stratified split 
X = df.drop("AdoptionSpeed", axis=1)
y = df["AdoptionSpeed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

#We store the partiions of the final df
X_train.to_parquet("../data/cleaned/X_train.parquet", index=False)
X_test.to_parquet("../data/cleaned/X_test.parquet", index=False)
y_train.to_frame().to_parquet("../data/cleaned/y_train.parquet", index=False)
y_test.to_frame().to_parquet("../data/cleaned/y_test.parquet", index=False)
'''

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## Dataset loading

In [46]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## Baselines

scoring for all the models

In [ ]:
    qwk = make_scorer(cohen_kappa_score, weights="quadratic")
    used_scores = {
            "f1_macro": "f1_macro",
            "QWK": qwk
        }

## RandomForest Baseline with original csv only deleting NA rows without joining with JSONS

In [43]:
original_df = pd.read_csv("../data/data.csv")

#print(original_df.columns.tolist())

In [44]:
print(f"Shape: {original_df.shape}")
print(f"NA Values per columns {original_df.isna().sum()}")

#we delete rows with NA for this quick baseline model with original data 

original_df = original_df.dropna(axis=0)

print(f"Shape: {original_df.shape}")
print(f"NA Values per columns {original_df.isna().sum()}")


#We also delete the animal names, description, PetId
original_df.drop(columns=["Name"],inplace=True)
original_df.drop(columns=["Description"],inplace=True)
original_df.drop(columns=["PetID"],inplace=True)

all_categorical = orig_onehot_enc_cols + orig_target_enc_cols

original_df[all_categorical] = original_df[all_categorical].astype(str)


print(original_df.columns.tolist())




#Data split
X = original_df.drop("AdoptionSpeed", axis=1)
y = original_df["AdoptionSpeed"]
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)



Shape: (14993, 24)
NA Values per columns Type                0
Name             1265
Age                 0
Breed1              0
Breed2              0
Gender              0
Color1              0
Color2              0
Color3              0
MaturitySize        0
FurLength           0
Vaccinated          0
Dewormed            0
Sterilized          0
Health              0
Quantity            0
Fee                 0
State               0
RescuerID           0
VideoAmt            0
Description        13
PetID               0
PhotoAmt            0
AdoptionSpeed       0
dtype: int64
Shape: (13716, 24)
NA Values per columns Type             0
Name             0
Age              0
Breed1           0
Breed2           0
Gender           0
Color1           0
Color2           0
Color3           0
MaturitySize     0
FurLength        0
Vaccinated       0
Dewormed         0
Sterilized       0
Health           0
Quantity         0
Fee              0
State            0
RescuerID        0
VideoAmt        

In [ ]:

baseline_clf = RandomForestClassifier(random_state=42, class_weight="balanced")

pipe = Pipeline (steps=[("preprocessor",originaldf_preprocessor), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train_orig, y_train_orig, cv=skf, scoring=used_scores) 

print(f"F1-macro: {round(np.mean(scores['test_f1_macro']),2)}, QWK mean: {round(np.mean(scores['test_QWK']),2)}, QWK std: {round(np.std(scores['test_QWK']),2)}")


"""
RESULTS: F1-macro: 0.32, QWK mean: 0.32, QWK std: 0.01
"""

F1-macro: 0.32, QWK mean: 0.32, QWK std: 0.01


## Random Baseline (stratified)

In [ ]:
#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = DummyClassifier(random_state=42,strategy="stratified")

pipe = Pipeline (steps=[("preprocessor",preprocessor), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores) 

print(f"F1-macro: {round(np.mean(scores['test_f1_macro']),2)}, QWK mean: {round(np.mean(scores['test_QWK']),2)}, QWK std: {round(np.std(scores['test_QWK']),2)}")

"""
Results: F1-macro: 0.2, QWK mean: -0.0, QWK std: 0.02
"""

F1-macro: 0.2, QWK mean: -0.0, QWK std: 0.02


## Random Baseline (most_frequent)

In [ ]:
#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = DummyClassifier(random_state=42, strategy="most_frequent")

pipe = Pipeline (steps=[("preprocessor",preprocessor), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores) 

print(f"F1-macro: {round(np.mean(scores['test_f1_macro']),2)}, QWK mean: {round(np.mean(scores['test_QWK']),2)}, QWK std: {round(np.std(scores['test_QWK']),2)}")

"""
Results: F1-macro: 0.09, QWK mean: 0.0, QWK std: 0.0
"""

F1-macro: 0.09, QWK mean: 0.0, QWK std: 0.0


## RandomForest Baseline

In [55]:
#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = RandomForestClassifier(random_state=42, class_weight="balanced")

pipe = Pipeline (steps=[("preprocessor",preprocessor), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores) 

print(f"F1-macro: {round(np.mean(scores['test_f1_macro']),2)}, QWK mean: {round(np.mean(scores['test_QWK']),2)}, QWK std: {round(np.std(scores['test_QWK']),2)}")

#TODO HAY QUE EVALUAR TAMBIEN CON EL TEST SET, Y ESE SERA NUESTRO BASELINE DE PUNTUACION REAL

"""
RESULTS 
-->class_weight balanced: F1-macro: 0.34, QWK mean: 0.37, QWK std: 0.02
-->class_weight default: F1-macro: 0.34, QWK mean: 0.36, QWK std: 0.02

"""

F1-macro: 0.34, QWK mean: 0.37, QWK std: 0.02


'\nRESULTS \n-->class_weight balanced: F1-macro: 0.34, QWK mean: 0.37, QWK std: 0.02\n-->class_weight default: F1-macro: 0.34, QWK mean: 0.36, QWK std: 0.02\n\n'